<a href="https://colab.research.google.com/github/ced-sys/earthquakes-and-Python/blob/main/Monte_Carlo_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Monte Carlo Earthquake simulations: 10,000 Years odf Seismic History
#Advanced probabilistic modelling of earthquake occurence and ground motion
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import poisson, lognorm, truncnorm
import warnings
warnings.filterwarnings('ignore')

#Set random seed for reproducibility
np.random.seed(42)

In [ ]:
#Fault Parameters -Laikipia-Marmanet Fault System
FAULT_NAME="Laikipia-Marmanet"
FAULT_LENGTH_KM=40.0 #Strike length (km)
FAULT_WIDTH_KM=15.0 #Down-sip width (km)
FAULT_AREA_KM2=FAULT_LENGTH_KM*FAULT_WIDTH_KM

#Slip rate Parameters (with uncertainty)
SLIP_RATE_MEAN=0.5 #mm/year (mean)
SLIP_RATE_STD=0.15 #mm/year (standard deviation)
SLIP_PER_EVENT_MEAN=1.0 #Meters per event (mean)
SLIP_PER_EVENT_STD=0.3 #Meters per event (std)

#Simulation parameters
SIMULATION_YEARS=10000 #Years to simulate
N_REALIZATIONS=1000 #Number of Monte Carlo realizations
SITE_DISTANCES=[10, 20, 30, 50, 100] #Site distances

#Seismic Parameters
SHEAR_MODULUS=30e9 #Pa
CRUSTAL_DENSITY=2700 #kg/m3
BETA=2.3 #Gutenberg-Richter b-value

print(f" Fault System: {FAULT_NAME}")
print(f" Fault Dimensions: {FAULT_LENGTH_KM} x {FAULT_WIDTH_KM} km")
print(f" Monte Carlo Realizations: {N_REALIZATIONS}")
print(f" Simulation Period: {SIMULATION_YEARS} years")

In [ ]:
def boore_atkinson_gmpe(mw, r_hypo, vs30=760):
  """
  Boore-Atkinson GMPE for PGA estimation

  Parameters:

  mw:float // Moment magnitude
  r_hypo: float //Hypocentral distance (km)
  vs30: float // Average shear wave velocity i top 30m (m/s)

  Returns:
  pga: float // Peak Ground Acceleration (g)
  """

  #Simplified Boore-Atkinson coefficients
  c1, c2, c3, c4=-1.715, 0.954, -1.255, 0.18

  #Calculate PGA
  log_pga=(c1+c2*mw+c3*np.log10(r_hypo+10)+
           c4*np.log10(vs30/760))

  return 10** log_pga

In [ ]:
def campbell_bozorgnia_gmpe(mw, r_rup, vs30=760):
  """
  Campbell-Bozorgnia GMPE for PGA estimation

  Parameters:
  mw: float // moment magnitude
  r_rup: float // Rupture distance (km)
  vs30: float // Average shear wave velocity (m/s)

  Returns:
  pga: float // Peak Ground Acceleration
  """

  #Simplified Campbell-Bozorgnia coefficients
  c0, c1, c2, c3=-1.715, 0.954, -1.255, 0.18

  log_pga=(c0+c1*mw+c2*np.log(r_rup+10)+
           c3 * np.log(vs30/760))

  return np.exp(log_pga)/981 #Convert to g

In [ ]:
#Magnitude Scaling relationships
def wells_coppersmith_magnitude(area_km2, fault_type='strike_slip'):
  """
  Wells and Coppersmith (1994) magnitude scaling

  Parameters:
  area_km2: float // Rupture area in km2
  fault_type: str // Type of fault ('strike_slip', 'reverse', 'normal')

  Returns:
  mw:float // Moment magnitde
  """

  coefficients={
      'strike_slip':{'a':4.07, 'b':0.98, 'sigma':0.24},
      'reverse':{'a':4.33, 'b':0.90, 'sigma':0.25},
      'normal':{'a':3.93, 'b':1.02, 'sigma':0.23}
  }

  coeff=coefficients[fault_type]
  mw=coeff['a']+coeff['b']*np.log10(area_km2)

  return mw, coeff['sigma']

In [ ]:
def leonard_magnitude(area_km2):
  """
  Leonard (2010) magnitude scaling for stable continental regions

  Parameters:
  area_km2: float // Rupture are in km2

  Returns:
  mw:float //Moment magnitude
  """
  if area_km2<537:
    mw=4.18+0.79*p.log10(area_km2)
  else:
    mw=3.99+0.98*np.log10(area_km2)

  return mw

In [ ]:
#Monte Carlo Simulation Engine
class EarthquakeMonteCarloSimulator:
  """
  Monte Carlo simulator for earthquake occurence and ground motion
  """

  def __init__(self, fault_params, simulation_params):
    self.fault_params=fault_params
    self.simulation_params=simulation_params
    self.catalogs=[]
    self.summary_stats=[]

  def simulate_single_realization(self, realization_id):
    """
    Simulate a single realization of earthquake activity

    Parameters:
    realization_id: int //Unique Identifier for this realization

    Returns:
    catalog: pdDataFrame // Earthquake catalog for this realization
    """
    #Sample uncertain parameters
    slip_rate=np.random.normal(SLIP_RATE_MEAN, SLIP_RATE_STD)
    slip_rate=max(0.1, slip_rate) #Minimum 0.1 mm/year

    slip_per_event=np.random.normal(SLIP_PER_EVENT_MEAN, SLIP_PER_EVENT_STD)
    slip_per_event=max(0.1, slip_per_event) #minimum 0.1 m

    #Calculate earthquake rate
    total_slip=(slip_rate* SIMULATION_YEARS) / 1000 #Convert to meters
    expected_events=total_slip/slip_per_event

    #Use Poisson distribution for number of events
    n_events=np.random.poisson(expected_events)

    if n_events==0:
      return pd.DataFrame()

    #Generate earthquake times (uniform distribution over 10,000 years)
    event_times=np.random.uniform(0, SIMULATION_YEARS, n_events)
    event_times=np.sort(event_times)

    #Calculate magnitudes using scaling relationships
    mw_wc, sigma_wc=wells_coppersmith_magnitude(FAULT_AREA_KM2)
    mw_leonard=leonard_magnitude(FAULT_AREA_KM2)

    #Use Wells & Coppersmith as primary,with uncertainty
    magnitudes=np.random.normal(mw_wc, sigma_wc, n_events)
    magnitudes=np.clip(magnitudes, 5.0, 8.5) #Reasonable bounds

    #Calculate seismic moments
    moments=10**(1.5* magnitudes+9.1) #Hanks-Kanamori

    #Generate ground motions for different sites
    catalog_data=[]

    for i, (time, mw, moment) in enumerate(zip(event_times, magnitudes, moments)):
      #Calculate hypocentral distance (with uncertainty)
      depth=np.random.uniform(5, 15) #km

      for site_dist in SITE_DISTANCES:
        r_hypo=np.sqrt(site_dist**2 + depth**2)

        #Calculate PGS using multiple GMPEs
        pga_ba=boore_atkinson_gmpe(mw, r_hypo)
        pga_cb=campbell_bozorgnia_gmpe(mw, r_hypo)

        #Add aleatory uncertainty
        pga_ba *= np.random.lognormal(0, 0.3)
        pga_cb*= np.random.lognormal(0, 0.3)

        catalog_data.append({
            'realization': realization_id,
            'event_id':i,
            'year':time,
            'magnitude':mw,
            'moment': moment,
            'depth':depth,
            'site_distance': site_dist,
            'hypo_distance': r_hypo,
            'pga_boore_atkinson':pga_ba,
            'pga_campbell_bozorgnia':pga_cb,
            'pga_mean':(pga_ba+pga_cb)/2,
            'slip_rate':slip_rate,
            'slip_per_event':slip_per_event
        })

    return pd.DataFrame(catalog_data)

  def run_monte_carlo(self):
    """
    Run the full Monte Carlo simulation
    """
    print("\nRunning Monte Carlo Simulation...")
    all_catalogs=[]

    for i in range(N_REALIZATIONS):
      if (i+1)% 100 ==0:
        print(f"Completed {i+1}/{N_REALIZATIONS} realizations")

      catalog=self.simulate_single_realization(i)
      if not catalog.empty:
        all_catalogs.append(catalog)

    #Combine all catalogs
    self.full_catalog=pd.concat(all_catalogs, ignore_index=True)

    print(f"Simulation complete")
    print(f"Total simulated events: {len(self.full_catalog)}")
    print(f"Events per realization: {len(self.full_catalog)/ N_REALIZATIONS:.1f}")

    return self.full_catalog

  def calculate_summary_statistics(self):
    """
    Calculate summary statistics across all realizations
    """
    stats_by_distance=[]

    for distance in SITE_DISTANCES:
      subset=self.full_catalog[self.full_catalog['site_distance']== distance]

      if len(subset)>0:
        stats={
            'site_distance':distance,
            'n_events':len(subset),
            'mean_magnitude': subset['magnitude'].mean(),
            'std_magnitude':subset['magnitude'].std(),
            'min_magnitude':subset['magnitude'].min(),
            'max_magnitude':subset['magnitude'].max(),
            'mean_pga':subset['pga_mean'].mean(),
            'std_pga':subset['pga_mean'].std(),
            'pga_95th': subset['pga_mean'].quantile(0.95),
            'pga_99th':subset['pga_mean'].quantile(0.99),
            'events_per_1000_years':len(subset)/N_REALIZATIONS* 1000/ SIMULATION_YEARS
        }
        stats_by_distance.append(stats)

    self.summary_stats=pd.DataFrame(stats_by_distance)
    return self.summary_stats

In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7)
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Magnitude Distribution (10,000 Years)')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
for distance in SITE_DISTANCES[:3]:
   #Show the first 3 distances
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True)

plt.title('PGA Distribution by Distance')
plt.xlabel('Peak Ground Acceleration (g)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Site Distance (km)')
plt.title('Magnitude vs PGA')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Peak Ground Acceleration (g)')
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, 'o-', label='Simulated Data')
plt.title('Gutenberg-Richter Relationship')
plt.xlabel('Magnitude (Mw)')
plt.ylabel('Cumulative Number of Events')
plt.grid(True, alpha=0.3)
plt.legend()

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Temporal Distribution of Events')
plt.xlabel('Years Before Present')
plt.ylabel('Number of Events')
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.6)
plt.title('Seismic Moment vs magnitude')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Seismic Moment (N-m)')
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GNPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.6, c=sample_data['magnitude'], cmap='plasma')
plt.colorbar(label='Magnitude (Mw)')
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison')
plt.xlabel('Boore-Atkinson PGA (g)')
plt.ylabel('Campbell-Bozorgnia PGA (g)')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

#8. Distance attenuation
plt.subplot(4, 3, 8)
for mag_bin in [5.5, 6.0, 6.5, 7.0]:
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, 'o--',
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation')
plt.xlabel('Site Distance (km)')
plt.ylabel('Mean PGA (g)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

#9. Event rate by magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.7, edgecolor='black')
plt.title('Event Rate by Magnitude')
plt.xlabel('Magnitude (Mw)')
plt.ylabel('Events per 1000 Years')
plt.grid(True, alpha=0.3)

#10. Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
for distance in SITE_DISTANCES:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, 'o--',
               label=f'{distance} km')

plt.title('Seismic hazard curves')
plt.xlabel('Peak Ground Acceleration (g)')
plt.ylabel('Annual Exceedance Probability')
plt.legend()
plt.grid(True, alpha=0.3)

#11. Summary stats table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
plt.title('Return Period Analysis')
plt.xlabel('Return Period (Years)')
plt.ylabel('PGA (g)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")


#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

# Task
Improve the legibility, size, readability, and aesthetics of the plots in the provided code.

## Increase figure size

### Subtask:
Modify the figure size to accommodate larger plots and better readability.


**Reasoning**:
Adjust the figure size for better readability of the plots.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))


#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7)
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Magnitude Distribution (10,000 Years)')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
for distance in SITE_DISTANCES[:3]:
   #Show the first 3 distances
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True)

plt.title('PGA Distribution by Distance')
plt.xlabel('Peak Ground Acceleration (g)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Site Distance (km)')
plt.title('Magnitude vs PGA')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Peak Ground Acceleration (g)')
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, 'o-', label='Simulated Data')
plt.title('Gutenberg-Richter Relationship')
plt.xlabel('Magnitude (Mw)')
plt.ylabel('Cumulative Number of Events')
plt.grid(True, alpha=0.3)
plt.legend()

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Temporal Distribution of Events')
plt.xlabel('Years Before Present')
plt.ylabel('Number of Events')
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.6)
plt.title('Seismic Moment vs magnitude')
plt.xlabel('Moment Magnitude (Mw)')
plt.ylabel('Seismic Moment (N-m)')
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GNPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.6, c=sample_data['magnitude'], cmap='plasma')
plt.colorbar(label='Magnitude (Mw)')
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison')
plt.xlabel('Boore-Atkinson PGA (g)')
plt.ylabel('Campbell-Bozorgnia PGA (g)')
plt.xscale('log')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

#8. Distance attenuation
plt.subplot(4, 3, 8)
for mag_bin in [5.5, 6.0, 6.5, 7.0]:
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, 'o--',
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation')
plt.xlabel('Site Distance (km)')
plt.ylabel('Mean PGA (g)')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)

#9. Event rate by magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.7, edgecolor='black')
plt.title('Event Rate by Magnitude')
plt.xlabel('Magnitude (Mw)')
plt.ylabel('Events per 1000 Years')
plt.grid(True, alpha=0.3)

#10. Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
for distance in SITE_DISTANCES:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, 'o--',
               label=f'{distance} km')

plt.title('Seismic hazard curves')
plt.xlabel('Peak Ground Acceleration (g)')
plt.ylabel('Annual Exceedance Probability')
plt.legend()
plt.grid(True, alpha=0.3)

#11. Summary stats table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
plt.title('Summary Statistics')


plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")


#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Adjust font sizes

### Subtask:
Increase title, label, and legend font sizes for improved legibility.


**Reasoning**:
Increase the font size of titles, labels, and legends in all plots, and the text in the summary statistics table to improve readability as requested in the subtask.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))


#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7)
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Magnitude Distribution (10,000 Years)', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
for distance in SITE_DISTANCES[:3]:
   #Show the first 3 distances
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True)

plt.title('PGA Distribution by Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Site Distance (km)', fontsize=12)
plt.title('Magnitude vs PGA', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, 'o-', label='Simulated Data')
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years Before Present', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.6)
plt.title('Seismic Moment vs magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GNPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.6, c=sample_data['magnitude'], cmap='plasma')
plt.colorbar(label='Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance attenuation
plt.subplot(4, 3, 8)
for mag_bin in [5.5, 6.0, 6.5, 7.0]:
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, 'o--',
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event rate by magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.7, edgecolor='black')
plt.title('Event Rate by Magnitude', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
for distance in SITE_DISTANCES:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, 'o--',
               label=f'{distance} km')

plt.title('Seismic hazard curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary stats table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center',
                fontsize=10) # Increased font size for table text
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)


plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")


#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

**Reasoning**:
The previous code failed because `fontsize` is not a valid argument for `plt.colorbar()`. The font size of the colorbar label needs to be set using `set_label` method with the `fontsize` argument. I will fix this error and regenerate the plots with the specified font sizes for all elements, including the colorbar label and the table text.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))


#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7)
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Magnitude Distribution (10,000 Years)', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
for distance in SITE_DISTANCES[:3]:
   #Show the first 3 distances
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True)

plt.title('PGA Distribution by Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.6)
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12) # Set colorbar label fontsize separately
plt.title('Magnitude vs PGA', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, 'o-', label='Simulated Data')
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years Before Present', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.6)
plt.title('Seismic Moment vs magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GNPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.6, c=sample_data['magnitude'], cmap='plasma')
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12) # Set colorbar label fontsize separately
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance attenuation
plt.subplot(4, 3, 8)
for mag_bin in [5.5, 6.0, 6.5, 7.0]:
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, 'o--',
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event rate by magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.7, edgecolor='black')
plt.title('Event Rate by Magnitude', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
for distance in SITE_DISTANCES:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, 'o--',
               label=f'{distance} km')

plt.title('Seismic hazard curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary stats table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center',
                fontsize=10) # Increased font size for table text
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)


plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")


#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Refine titles and labels

### Subtask:
Ensure clear and concise titles and labels for each plot.


**Reasoning**:
Review and improve the titles and labels for clarity and conciseness in the plots.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))

#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7)
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Simulated Earthquake Magnitude Distribution', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency of Events', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
for distance in SITE_DISTANCES[:3]:
   #Show the first 3 distances
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True)

plt.title('PGA Distribution by Site Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. Magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.6)
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12)
plt.title('Magnitude vs Peak Ground Acceleration', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-Richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, 'o-', label='Simulated Data')
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black')
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs Magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.6)
plt.title('Seismic Moment vs Magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GMPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.6, c=sample_data['magnitude'], cmap='plasma')
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance Attenuation
plt.subplot(4, 3, 8)
for mag_bin in [5.5, 6.0, 6.5, 7.0]:
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, 'o--',
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation of Mean PGA', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event Rate by Magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.7, edgecolor='black')
plt.title('Event Rate by Magnitude Bin', fontsize=14)
plt.xlabel('Magnitude Bin (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Seismic Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
for distance in SITE_DISTANCES:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, 'o--',
               label=f'{distance} km')

plt.title('Seismic Hazard Curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary Statistics Table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)


plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")


#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Improve plot aesthetics

### Subtask:
Enhance the visual appeal of the plots by adjusting elements like line styles, markers, colors, and potentially adding annotations or text.


**Reasoning**:
Enhance the visual appeal of the plots by adjusting line styles, markers, colors, and transparency to improve visual distinction and readability.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis") # Changed palette for better visual distinction

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))

#1. Magnitude Distribution
plt.subplot(4, 3, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7, color='skyblue') # Added color
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Simulated Earthquake Magnitude Distribution', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency of Events', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(4, 3, 2)
colors = sns.color_palette("viridis", len(SITE_DISTANCES[:3])) # Define colors for distances
for i, distance in enumerate(SITE_DISTANCES[:3]):
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True, color=colors[i]) # Added color

plt.title('PGA Distribution by Site Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. Magnitude vs PGA
plt.subplot(4, 3, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.7, s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12)
plt.title('Magnitude vs Peak Ground Acceleration', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-Richter Relationship
plt.subplot(4, 3, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, marker='o', linestyle='-', label='Simulated Data', color='purple') # Added marker, linestyle, and color
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(4, 3, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black', color='lightcoral') # Added color
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs Magnitude
plt.subplot(4, 3, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.7, color='teal', s=10) # Added color and adjusted size
plt.title('Seismic Moment vs Magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GMPE Comparison
plt.subplot(4, 3, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.7, c=sample_data['magnitude'], cmap='plasma', s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance Attenuation
plt.subplot(4, 3, 8)
markers = ['o', 's', '^', 'D'] # Define markers for magnitude bins
linestyles = ['-', '--', '-.', ':'] # Define linestyles
colors = sns.color_palette("viridis", len([5.5, 6.0, 6.5, 7.0])) # Define colors

for i, mag_bin in enumerate([5.5, 6.0, 6.5, 7.0]):
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, marker=markers[i], linestyle=linestyles[i], color=colors[i],
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation of Mean PGA', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event Rate by Magnitude
plt.subplot(4, 3, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.8, edgecolor='black', color='forestgreen') # Increased alpha and added color
plt.title('Event Rate by Magnitude Bin', fontsize=14)
plt.xlabel('Magnitude Bin (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Seismic Hazard Curves
plt.subplot(4, 3, 10)
pga_levels=np.logspace(-3, 0, 30)
markers = ['o', 's', '^', 'D', 'v'] # Define markers for distances
linestyles = ['-', '--', '-.', ':', '-'] # Define linestyles
colors = sns.color_palette("viridis", len(SITE_DISTANCES)) # Define colors

for i, distance in enumerate(SITE_DISTANCES):
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, marker=markers[i], linestyle=linestyles[i], color=colors[i],
               label=f'{distance} km')

plt.title('Seismic Hazard Curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary Statistics Table
plt.subplot(4, 3, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)

plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")

#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Arrange subplots

### Subtask:
Optimize the layout of subplots for better use of space and flow.


**Reasoning**:
I need to optimize the layout of the subplots to improve the use of space and flow. The current layout is a 4x3 grid with 11 plots, leaving one empty space. I will change the grid to a 3x4 layout, which is a better fit for 11 plots and allows for a more logical arrangement. I will also adjust the subplot indices accordingly and ensure the summary statistics table is still visible.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FALT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis") # Changed palette for better visual distinction

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))

# Change subplot grid to 3x4 and rearrange plots

#1. Magnitude Distribution
plt.subplot(3, 4, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7, color='skyblue') # Added color
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Simulated Earthquake Magnitude Distribution', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency of Events', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(3, 4, 2)
colors = sns.color_palette("viridis", len(SITE_DISTANCES[:3])) # Define colors for distances
for i, distance in enumerate(SITE_DISTANCES[:3]):
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True, color=colors[i]) # Added color

plt.title('PGA Distribution by Site Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. Magnitude vs PGA
plt.subplot(3, 4, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.7, s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12)
plt.title('Magnitude vs Peak Ground Acceleration', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-Richter Relationship
plt.subplot(3, 4, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, marker='o', linestyle='-', label='Simulated Data', color='purple') # Added marker, linestyle, and color
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(3, 4, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black', color='lightcoral') # Added color
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs Magnitude
plt.subplot(3, 4, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.7, color='teal', s=10) # Added color and adjusted size
plt.title('Seismic Moment vs Magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GMPE Comparison
plt.subplot(3, 4, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.7, c=sample_data['magnitude'], cmap='plasma', s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance Attenuation
plt.subplot(3, 4, 8)
markers = ['o', 's', '^', 'D'] # Define markers for magnitude bins
linestyles = ['-', '--', '-.', ':'] # Define linestyles
colors = sns.color_palette("viridis", len([5.5, 6.0, 6.5, 7.0])) # Define colors

for i, mag_bin in enumerate([5.5, 6.0, 6.5, 7.0]):
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, marker=markers[i], linestyle=linestyles[i], color=colors[i],
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation of Mean PGA', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event Rate by Magnitude
plt.subplot(3, 4, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.8, edgecolor='black', color='forestgreen') # Increased alpha and added color
plt.title('Event Rate by Magnitude Bin', fontsize=14)
plt.xlabel('Magnitude Bin (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Seismic Hazard Curves
plt.subplot(3, 4, 10)
pga_levels=np.logspace(-3, 0, 30)
markers = ['o', 's', '^', 'D', 'v'] # Define markers for distances
linestyles = ['-', '--', '-.', ':', '-'] # Define linestyles
colors = sns.color_palette("viridis", len(SITE_DISTANCES)) # Define colors

for i, distance in enumerate(SITE_DISTANCES):
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, marker=markers[i], linestyle=linestyles[i], color=colors[i],
               label=f'{distance} km')

plt.title('Seismic Hazard Curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary Statistics Table
plt.subplot(3, 4, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)

plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")

#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

**Reasoning**:
The previous command failed because of a typo in the variable name `FALT_AREA_KM2`. I need to correct this to `FAULT_AREA_KM2` and re-run the code to apply the updated subplot layout.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis") # Changed palette for better visual distinction

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))

# Change subplot grid to 3x4 and rearrange plots

#1. Magnitude Distribution
plt.subplot(3, 4, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7, color='skyblue') # Added color
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Simulated Earthquake Magnitude Distribution', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency of Events', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#2. PGA Distribution by Distance
plt.subplot(3, 4, 2)
colors = sns.color_palette("viridis", len(SITE_DISTANCES[:3])) # Define colors for distances
for i, distance in enumerate(SITE_DISTANCES[:3]):
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True, color=colors[i]) # Added color

plt.title('PGA Distribution by Site Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

#3. Magnitude vs PGA
plt.subplot(3, 4, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.7, s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12)
plt.title('Magnitude vs Peak Ground Acceleration', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#4. Gutenberg-Richter Relationship
plt.subplot(3, 4, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, marker='o', linestyle='-', label='Simulated Data', color='purple') # Added marker, linestyle, and color
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

#5. Temporal Distribution
plt.subplot(3, 4, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black', color='lightcoral') # Added color
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)

#6. Seismic Moment vs Magnitude
plt.subplot(3, 4, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.7, color='teal', s=10) # Added color and adjusted size
plt.title('Seismic Moment vs Magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)

#7. GMPE Comparison
plt.subplot(3, 4, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.7, c=sample_data['magnitude'], cmap='plasma', s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#8. Distance Attenuation
plt.subplot(3, 4, 8)
markers = ['o', 's', '^', 'D'] # Define markers for magnitude bins
linestyles = ['-', '--', '-.', ':'] # Define linestyles
colors = sns.color_palette("viridis", len([5.5, 6.0, 6.5, 7.0])) # Define colors

for i, mag_bin in enumerate([5.5, 6.0, 6.5, 7.0]):
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, marker=markers[i], linestyle=linestyles[i], color=colors[i],
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation of Mean PGA', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#9. Event Rate by Magnitude
plt.subplot(3, 4, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.8, edgecolor='black', color='forestgreen') # Increased alpha and added color
plt.title('Event Rate by Magnitude Bin', fontsize=14)
plt.xlabel('Magnitude Bin (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)

#10. Seismic Hazard Curves
plt.subplot(3, 4, 10)
pga_levels=np.logspace(-3, 0, 30)
markers = ['o', 's', '^', 'D', 'v'] # Define markers for distances
linestyles = ['-', '--', '-.', ':', '-'] # Define linestyles
colors = sns.color_palette("viridis", len(SITE_DISTANCES)) # Define colors

for i, distance in enumerate(SITE_DISTANCES):
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, marker=markers[i], linestyle=linestyles[i], color=colors[i],
               label=f'{distance} km')

plt.title('Seismic Hazard Curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

#11. Summary Statistics Table
plt.subplot(3, 4, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)

plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")

#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Review and iterate

### Subtask:
Review the updated plots and make further adjustments as needed to achieve the desired legibility and aesthetic.


**Reasoning**:
Examine the generated plots and identify any remaining areas for improvement in terms of legibility and aesthetics, then generate the code to make those final adjustments.



In [ ]:
#Initialize simulator
simulator=EarthquakeMonteCarloSimulator(
    fault_params={'name': FAULT_NAME, 'area': FAULT_AREA_KM2},
    simulation_params={'years':SIMULATION_YEARS, 'realizations': N_REALIZATIONS}
)

#Run Monte Carlo Simulation
catalog=simulator.run_monte_carlo()

#Calculate summary statistics
summary_stats=simulator.calculate_summary_statistics()

#Visualization and analysis

#Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis") # Changed palette for better visual distinction

# Adjust the figure size to accommodate more plots and better readability
plt.figure(figsize=(20, 15))

# Change subplot grid to 3x4 and rearrange plots

#1. Magnitude Distribution
plt.subplot(3, 4, 1)
sns.histplot(catalog['magnitude'], bins=30, kde=True, alpha=0.7, color='skyblue') # Added color
plt.axvline(catalog['magnitude'].mean(), color='red', linestyle='--',
            label=f'Mean: {catalog["magnitude"].mean():.2f}')
plt.title('Simulated Earthquake Magnitude Distribution', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Frequency of Events', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#2. PGA Distribution by Distance
plt.subplot(3, 4, 2)
colors = sns.color_palette("viridis", len(SITE_DISTANCES[:3])) # Define colors for distances
for i, distance in enumerate(SITE_DISTANCES[:3]):
   subset=catalog[catalog['site_distance']==distance]
   sns.histplot(subset['pga_mean'], bins=30, alpha=0.6,
                label=f'{distance}km', kde=True, color=colors[i]) # Added color

plt.title('PGA Distribution by Site Distance', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#3. Magnitude vs PGA
plt.subplot(3, 4, 3)
sample_data=catalog.sample(n=min(2000, len(catalog)))
scatter=plt.scatter(sample_data['magnitude'],sample_data['pga_mean'],
                    c=sample_data['site_distance'], cmap='viridis', alpha=0.7, s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(scatter)
cbar.set_label('Site Distance (km)', fontsize=12)
plt.title('Magnitude vs Peak Ground Acceleration', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Peak Ground Acceleration (g)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#4. Gutenberg-Richter Relationship
plt.subplot(3, 4, 4)
magnitude_bins=np.arange(5.0, 8.5, 0.1)
n_events=[]
for mag in magnitude_bins:
  n_events.append(len(catalog[catalog['magnitude']>=mag]))

plt.semilogy(magnitude_bins, n_events, marker='o', linestyle='-', label='Simulated Data', color='purple') # Added marker, linestyle, and color
plt.title('Gutenberg-Richter Relationship', fontsize=14)
plt.xlabel('Magnitude (Mw)', fontsize=12)
plt.ylabel('Cumulative Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#5. Temporal Distribution
plt.subplot(3, 4, 5)
plt.hist(catalog['year'], bins=50, alpha=0.7, edgecolor='black', color='lightcoral') # Added color
plt.title('Temporal Distribution of Events', fontsize=14)
plt.xlabel('Years', fontsize=12)
plt.ylabel('Number of Events', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#6. Seismic Moment vs Magnitude
plt.subplot(3, 4, 6)
plt.scatter(catalog['magnitude'], catalog['moment'], alpha=0.7, color='teal', s=10) # Added color and adjusted size
plt.title('Seismic Moment vs Magnitude', fontsize=14)
plt.xlabel('Moment Magnitude (Mw)', fontsize=12)
plt.ylabel('Seismic Moment (N-m)', fontsize=12)
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#7. GMPE Comparison
plt.subplot(3, 4, 7)
sample_data=catalog.sample(n=min(1000, len(catalog)))
plt.scatter(sample_data['pga_boore_atkinson'], sample_data['pga_campbell_bozorgnia'],
            alpha=0.7, c=sample_data['magnitude'], cmap='plasma', s=10) # Increased alpha and adjusted size
cbar = plt.colorbar(label='Magnitude (Mw)')
cbar.set_label('Magnitude (Mw)', fontsize=12)
plt.plot([0.001, 1], [0.001, 1], 'r--', label='1:1 Line')
plt.title('GMPE Comparison', fontsize=14)
plt.xlabel('Boore-Atkinson PGA (g)', fontsize=12)
plt.ylabel('Campbell-Bozorgnia PGA (g)', fontsize=12)
plt.xscale('log')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size


#8. Distance Attenuation
plt.subplot(3, 4, 8)
markers = ['o', 's', '^', 'D'] # Define markers for magnitude bins
linestyles = ['-', '--', '-.', ':'] # Define linestyles
colors = sns.color_palette("viridis", len([5.5, 6.0, 6.5, 7.0])) # Define colors

for i, mag_bin in enumerate([5.5, 6.0, 6.5, 7.0]):
  subset=catalog[
      (catalog['magnitude']>= mag_bin-0.25) &
      (catalog['magnitude']< mag_bin+0.25)
  ]

  if len(subset)>0:
    mean_pga=subset.groupby('site_distance')['pga_mean'].mean()
    plt.plot(mean_pga.index, mean_pga.values, marker=markers[i], linestyle=linestyles[i], color=colors[i],
             label=f'Mw {mag_bin}')

plt.title('Distance Attenuation of Mean PGA', fontsize=14)
plt.xlabel('Site Distance (km)', fontsize=12)
plt.ylabel('Mean PGA (g)', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#9. Event Rate by Magnitude
plt.subplot(3, 4, 9)
mag_bins=np.arange(5.0, 8.0, 0.2)
event_rates=[]
for i in range(len(mag_bins)-1):
  count=len(catalog[
      (catalog['magnitude']>= mag_bins[i])&
      (catalog['magnitude']< mag_bins[i+1])
  ])
  rate=count /N_REALIZATIONS /SIMULATION_YEARS * 1000 #Per 1000 years
  event_rates.append(rate)

plt.bar(mag_bins[:-1], event_rates, width=0.15, alpha=0.8, edgecolor='black', color='forestgreen') # Increased alpha and added color
plt.title('Event Rate by Magnitude Bin', fontsize=14)
plt.xlabel('Magnitude Bin (Mw)', fontsize=12)
plt.ylabel('Events per 1000 Years', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#10. Seismic Hazard Curves
plt.subplot(3, 4, 10)
pga_levels=np.logspace(-3, 0, 30)
markers = ['o', 's', '^', 'D', 'v'] # Define markers for distances
linestyles = ['-', '--', '-.', ':', '-'] # Define linestyles
colors = sns.color_palette("viridis", len(SITE_DISTANCES)) # Define colors

for i, distance in enumerate(SITE_DISTANCES):
  subset=catalog[catalog['site_distance']==distance]
  if len(subset) > 0:
    exceedance_probs=[]
    for pga_level in pga_levels:
      n_exceed=len(subset[subset['pga_mean']>= pga_level])
      prob=n_exceed/len(subset)
      exceedance_probs.append(prob)

    plt.loglog(pga_levels, exceedance_probs, marker=markers[i], linestyle=linestyles[i], color=colors[i],
               label=f'{distance} km')

plt.title('Seismic Hazard Curves', fontsize=14)
plt.xlabel('Peak Ground Acceleration (g)', fontsize=12)
plt.ylabel('Annual Exceedance Probability', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10) # Adjust tick label size

#11. Summary Statistics Table
plt.subplot(3, 4, 11)
plt.axis('tight')
plt.axis("off")
table_data=summary_stats.round(3)
table=plt.table(cellText=table_data.values,
                colLabels=table_data.columns,
                cellLoc='center',
                loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
plt.title('Summary Statistics', fontsize=14)

plt.tight_layout()
plt.show()

print("\nOverall Statistics:")
print(f"- Total simulated events: {len(catalog):,}")
print(f"-Mean events per realization: {len(catalog) / N_REALIZATIONS:.1f}")
print(f"-Mean magnitude: {catalog['magnitude'].mean():.2f} +/- {catalog['magnitude'].std():.2f}")
print(f"-Magnitude range: {catalog['magnitude'].min():.2f}-{catalog['magnitude'].max():.2f}")

print(f"\nSite Specific results:")
print(summary_stats.to_string(index=False))

print(f"\nReturn Period Analysis:")
for distance in [10, 30, 50]:
  subset=catalog[catalog['site_distance']==distance]
  if len(subset)>0:
    pga_10_year=subset['pga_mean'].quantile(0.9)
    pga_100_year=subset['pga_mean'].quantile(0.99)
    pga_475_year=subset['pga_mean'].quantile(0.998)

    print(f"- {distance} km distance:")
    print(f"- 10-year return period: {pga_10_year:.3f} g")
    print(f"- 100-year return period: {pga_100_year:.3f} g")
    print(f"- 475-year return period: {pga_475_year:.3f} g")

print(f"\nUncertainty Analysis:")
print(f"- Coefficient of variation (magnitude): {catalog['magnitude'].std()/ catalog['magnitude'].mean():.2f}")
print(f"- Coefficient of variation (PGA): {catalog['pga_mean'].std()/ catalog['pga_mean'].mean():.2f}")

#Export to results
catalog.to_csv('earthquake_catalog_10k_years.csv', index=False)
summary_stats.to_csv('summary_statistics.csv', index=False)

print(f"\nData Export:")
print(f"- Earthquake catalog: earthquake_catalog_10k_years.csv")
print(f"- Summary statistics: summary_statistics.csv")

print(f"\nMonte Carlo simulation completed successfully!")
print(f"{N_REALIZATIONS} realizations of {SIMULATION_YEARS} years each")
print(f"Total simulated time: {N_REALIZATIONS* SIMULATION_YEARS:,} years")

## Summary:

### Data Analysis Key Findings

*   The Monte Carlo simulation successfully generated earthquake catalogs and summary statistics over 10,000 years across 1000 realizations.
*   Initial plots were generated in a 4x3 grid with default font sizes and aesthetics.
*   Font sizes for titles, labels, legends, colorbar labels, and table text were successfully increased for improved readability.
*   Plot titles and axis labels were refined for greater clarity and conciseness.
*   Plot aesthetics were enhanced by adjusting color palettes, adding specific colors to plots, adjusting marker styles, and increasing alpha values for scatter plots.
*   The subplot layout was successfully changed from a 4x3 grid to a 3x4 grid, optimizing space utilization and visual flow.
*   Tick label sizes were adjusted to further improve legibility.
*   Data was exported to CSV files: 'earthquake\_catalog\_10k\_years.csv' and 'summary\_statistics.csv'.

### Insights or Next Steps

*   The improved plot legibility and aesthetics allow for clearer interpretation of the simulation results, which can aid in presenting the findings effectively.
*   Consider adding annotations to key features in the plots, such as specific return period values on the hazard curves or key statistics on the magnitude distribution, to further enhance readability and informativeness.
